# 🧠 Build & Push Your Own Storyteller SLM to Hugging Face

**What this notebook does:**
1. Loads 15,000 TinyStories + cleaned Gutenberg fairy tales
2. Fine-tunes `Qwen2.5-0.5B` using LoRA (efficient, GPU-friendly)
3. Merges weights and tests generation
4. Pushes the finished model to Hugging Face Hub

**Runtime:** Google Colab with T4 GPU (~30-45 mins total)

**Before you start:**
- Runtime → Change runtime type → **T4 GPU**
- Have your Hugging Face token ready (huggingface.co → Settings → Access Tokens)

---

## Step 1 · Install & Verify Environment

In [1]:

!pip install -q "huggingface_hub==1.12.2"
!pip install -q transformers datasets peft accelerate

import torch
import os, re, json, random
from pathlib import Path

print("=" * 50)
import huggingface_hub
print(f"huggingface_hub : {huggingface_hub.__version__}")
import datasets
print(f"datasets        : {datasets.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("=" * 50)
print("✅ Ready")


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


huggingface_hub : 1.12.2
datasets        : 5.0.0
PyTorch         : 2.12.1+cpu
GPU available   : False
✅ Ready


## Step 2 · Configuration — Set Everything Here

**Only this cell needs your attention before running the rest.**

In [3]:
"""
Central configuration for the entire training run.
Change values here — everything below uses these.
"""

# ── Model ────────────────────────────────────────────────────────────────────
# Qwen2.5-0.5B: best quality/speed balance for T4
# Alternative: "HuggingFaceTB/SmolLM2-360M" — slightly smaller, also good
BASE_MODEL = "Qwen/Qwen2.5-0.5B"

# ── Your Hugging Face details ─────────────────────────────────────────────────
HF_USERNAME   = "viviktchaudhary"        # my username
MODEL_REPO    = "tiny-slm-storyteller-v1"  #  name of  HF repo
HF_REPO_ID    = f"{HF_USERNAME}/{MODEL_REPO}"

# ── Dataset ───────────────────────────────────────────────────────────────────
TINYSTORIES_COUNT = 15_000   # stories from TinyStories dataset
GUTENBERG_URL     = True     # fetch Andersen fairy tales from Gutenberg
MAX_SEQ_LENGTH    = 256      # tokens per training example
CHUNK_STRIDE      = 128      # overlap between chunks (50%)

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS        = 2      # 2 is good — prevents overfitting
BATCH_SIZE        = 8       # per device
GRAD_ACCUMULATION = 2       # effective batch = 4×4 = 16
LEARNING_RATE     = 2e-4
WARMUP_STEPS      = 100

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR    = "./tiny_storyteller"

print("Configuration:")
print(f"  Base model      : {BASE_MODEL}")
print(f"  HF repo         : {HF_REPO_ID}")
print(f"  TinyStories     : {TINYSTORIES_COUNT:,} stories")
print(f"  Epochs          : {NUM_EPOCHS}")
print(f"  Effective batch : {BATCH_SIZE * GRAD_ACCUMULATION}")
print(f"  LoRA rank       : {LORA_R}")
print()



Configuration:
  Base model      : Qwen/Qwen2.5-0.5B
  HF repo         : viviktchaudhary/tiny-slm-storyteller-v1
  TinyStories     : 15,000 stories
  Epochs          : 2
  Effective batch : 16
  LoRA rank       : 16



## Step 3 · Build the Corpus

Two sources combined:
- **TinyStories** — 15,000 clean, structured short stories (perfect for small model training)
- **Project Gutenberg** — real Andersen fairy tales for linguistic richness and vocabulary depth

Both are wrapped with `<story>` tags so the model learns to produce complete stories with clear endings.

In [4]:
from datasets import load_dataset, Dataset
import urllib.request
import re

# ── Part A: TinyStories ───────────────────────────────────────────────────────

print(f"Loading {TINYSTORIES_COUNT:,} stories from TinyStories...")
tiny = load_dataset(
    "roneneldan/TinyStories",
    split=f"train[:{TINYSTORIES_COUNT}]"
)
print(f"  Loaded: {len(tiny):,} stories")
print(f"  Sample: {tiny[0]['text'][:320]}...")

# Wrap with story tags — teaches model where stories begin and end
tiny_texts = [f"<story>\n{row['text'].strip()}\n</story>" for row in tiny]


# ── Part B: Gutenberg Fairy Tales ────────────────────────────────────────────

def clean_gutenberg(raw_text):
    """
    Remove Project Gutenberg boilerplate, chapter headers, page numbers,
    and formatting artefacts from raw downloaded text.
    Returns clean prose only.
    """
    # Remove Gutenberg header and footer
    if '*** START OF' in raw_text:
        raw_text = raw_text[raw_text.index('*** START OF'):]
        raw_text = re.sub(r'\*{3}.*?\*{3}', '', raw_text, count=1)
    if '*** END OF' in raw_text:
        raw_text = raw_text[:raw_text.index('*** END OF')]
    # Remove chapter headings
    raw_text = re.sub(r'\n[A-Z][A-Z\s]+\.?\n', '\n', raw_text)
    # Remove page numbers
    raw_text = re.sub(r'\n\s*\d+\s*\n', '\n', raw_text)
    # Remove illustration notes
    raw_text = re.sub(r'\[Illustration.*?\]', '', raw_text)
    # Normalise whitespace
    raw_text = re.sub(r'\n{3,}', '\n\n', raw_text)
    return raw_text.strip()


def split_into_stories(text, min_words=80, max_words=400):
    """
    Split a long text into story-sized chunks by paragraph boundaries.
    Each chunk is wrapped with story tags.
    """
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 50]
    chunks, current = [], []
    current_words   = 0

    for para in paragraphs:
        words = len(para.split())
        if current_words + words > max_words and current_words >= min_words:
            story_text = ' '.join(current)
            chunks.append(f"<story>\n{story_text}\n</story>")
            current, current_words = [para], words
        else:
            current.append(para)
            current_words += words

    if current and current_words >= min_words:
        chunks.append(f"<story>\n{' '.join(current)}\n</story>")

    return chunks


# Gutenberg sources — Andersen fairy tales (public domain)
GUTENBERG_URLS = [
    "https://www.gutenberg.org/cache/epub/1597/pg1597.txt",   # Andersen vol 1
    "https://www.gutenberg.org/cache/epub/1598/pg1598.txt",   # Andersen vol 2
    "https://www.gutenberg.org/cache/epub/27200/pg27200.txt", # Grimm fairy tales
]

gutenberg_texts = []
if GUTENBERG_URL:
    print("\nFetching Gutenberg fairy tales...")
    for url in GUTENBERG_URLS:
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=15) as r:
                raw = r.read().decode('utf-8', errors='ignore')
            cleaned = clean_gutenberg(raw)
            stories = split_into_stories(cleaned)
            gutenberg_texts.extend(stories)
            print(f"  ✓ {url.split('/')[-1]} → {len(stories)} chunks")
        except Exception as e:
            print(f"  ✗ Failed ({url.split('/')[-1]}): {e}")
            print(f"    Continuing without this source...")

    print(f"  Total Gutenberg chunks: {len(gutenberg_texts)}")


# ── Combine ───────────────────────────────────────────────────────────────────
all_texts = tiny_texts + gutenberg_texts

import random
random.seed(42)
random.shuffle(all_texts)

print(f"\nFinal corpus:")
print(f"  TinyStories chunks : {len(tiny_texts):,}")
print(f"  Gutenberg chunks   : {len(gutenberg_texts):,}")
print(f"  Total              : {len(all_texts):,}")
print(f"  Approx words       : {sum(len(t.split()) for t in all_texts):,}")

Loading 15,000 stories from TinyStories...


README.md: 0.00B [00:00, ?B/s]

C:\Users\vivik\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vivik\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

  Loaded: 15,000 stories
  Sample: One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her m...

Fetching Gutenberg fairy tales...
  ✓ pg1597.txt → 1 chunks
  ✓ pg1598.txt → 1 chunks
  ✓ pg27200.txt → 1 chunks
  Total Gutenberg chunks: 3

Final corpus:
  TinyStories chunks : 15,000
  Gutenberg chunks   : 3
  Total              : 15,003
  Approx words       : 3,093,170


## Step 4 · Tokenise the Corpus

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
import torch

print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"
print(f"  Vocabulary size : {len(tokenizer):,}")
print(f"  EOS token       : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")


class StoryDataset(TorchDataset):
    """
    Tokenises all stories into fixed-length overlapping chunks.
    """
    def __init__(self, texts, tokenizer, chunk_size, stride):
        self.chunks = []
        print(f"Tokenising {len(texts):,} texts into {chunk_size}-token chunks...")

        for i, text in enumerate(texts):
            # === FIX: Prevent extremely long texts from causing warnings ===
            if len(text) > 80_000:          # Safe character limit
                text = text[:80_000]

            ids = tokenizer.encode(text, add_special_tokens=True)

            # Slide a window across the token sequence
            for start in range(0, max(1, len(ids) - chunk_size + 1), stride):
                chunk = ids[start : start + chunk_size]
                if len(chunk) < 32:   # skip very short chunks
                    continue
                # Pad to chunk_size
                pad_len = chunk_size - len(chunk)
                chunk   = chunk + [tokenizer.pad_token_id] * pad_len
                self.chunks.append(chunk)

            if (i + 1) % 2000 == 0:   # More frequent progress updates
                print(f"  Processed {i+1:,} / {len(texts):,} texts "
                      f"({len(self.chunks):,} chunks so far)")

        print(f"  Total training chunks: {len(self.chunks):,}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        ids = torch.tensor(self.chunks[idx], dtype=torch.long)
        return {
            "input_ids"      : ids,
            "attention_mask" : (ids != tokenizer.pad_token_id).long(),
            "labels"         : ids.clone(),
        }


train_dataset = StoryDataset(
    all_texts,
    tokenizer,
    chunk_size = MAX_SEQ_LENGTH,
    stride     = CHUNK_STRIDE,
)

print(f"\nDataset ready:")
print(f"  Training examples : {len(train_dataset):,}")
print(f"  Chunk size        : {MAX_SEQ_LENGTH} tokens")
print(f"  Stride            : {CHUNK_STRIDE} tokens (50% overlap)")

# Decode a sample to verify
sample = train_dataset[0]
decoded = tokenizer.decode(sample['input_ids'][:80], skip_special_tokens=False)
print(f"\nSample chunk (first 80 tokens):\n  {decoded}")

Loading tokenizer: Qwen/Qwen2.5-0.5B


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

C:\Users\vivik\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vivik\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Vocabulary size : 151,665
  EOS token       : '<|endoftext|>' (id=151643)
Tokenising 15,003 texts into 256-token chunks...
  Processed 2,000 / 15,003 texts (2,333 chunks so far)
  Processed 4,000 / 15,003 texts (4,563 chunks so far)
  Processed 6,000 / 15,003 texts (6,773 chunks so far)
  Processed 8,000 / 15,003 texts (9,305 chunks so far)
  Processed 10,000 / 15,003 texts (11,523 chunks so far)
  Processed 12,000 / 15,003 texts (13,748 chunks so far)
  Processed 14,000 / 15,003 texts (15,965 chunks so far)
  Total training chunks: 17,088

Dataset ready:
  Training examples : 17,088
  Chunk size        : 256 tokens
  Stride            : 128 tokens (50% overlap)

Sample chunk (first 80 tokens):
  <story>
Once upon a time, there was a huge bear named Ben. Ben loved to jog in the forest every day. One day, Ben met a little rabbit named Rosie.

"Hello, Rosie! Do you want to jog with me?" asked Ben.

"Oh no, I can't keep up with you. I'm too slow," replied Rosie.

"Don't worry, Rosie. If

## Step 5 · Load Model + Apply LoRA

LoRA (Low-Rank Adaptation) freezes the original model weights and adds small trainable matrices to the attention layers only. This means:
- **Trainable parameters: ~1% of total** — fast and memory efficient
- **Original weights preserved** — style learned, knowledge retained
- **Merge at the end** — final model has no LoRA dependency

In [6]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print(f"Loading base model: {BASE_MODEL}")
print("This takes 1-2 minutes on first run (downloading ~1GB)...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype  = torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map  = "auto",
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nBase model loaded:")
print(f"  Total parameters : {total_params:,}")


# Apply LoRA — only attention projection layers are trained
lora_config = LoraConfig(
    r               = LORA_R,
    lora_alpha      = LORA_ALPHA,
    target_modules  = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout    = LORA_DROPOUT,
    bias            = "none",
    task_type       = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable params : {trainable:,}  "
      f"({100*trainable/total_params:.2f}% of total)")
print(f"  Frozen params    : {total_params - trainable:,}")
print()
print(f"LoRA configuration:")
print(f"  Rank (r)         : {LORA_R}")
print(f"  Alpha            : {LORA_ALPHA}")
print(f"  Target modules   : q_proj, k_proj, v_proj, o_proj")
print()
print("✅ Model ready for training")

Loading base model: Qwen/Qwen2.5-0.5B
This takes 1-2 minutes on first run (downloading ~1GB)...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


Base model loaded:
  Total parameters : 494,032,768
  Trainable params : 2,162,688  (0.44% of total)
  Frozen params    : 491,870,080

LoRA configuration:
  Rank (r)         : 16
  Alpha            : 32
  Target modules   : q_proj, k_proj, v_proj, o_proj

✅ Model ready for training


## Step 6 · Train

Expected time: **20-30 minutes** on T4 GPU.

Watch the loss drop. A loss around **1.2–1.5** after training means the model has learned the story patterns well without memorising them.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import time

# Data collator handles the label shifting internally
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False,   # causal LM, not masked LM
)

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUMULATION,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = WARMUP_STEPS,
    lr_scheduler_type           = "cosine",
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 50,
    save_steps                  = 500,
    save_total_limit            = 1,
    report_to                   = "none",   # no wandb
    dataloader_num_workers      = 2,
    load_best_model_at_end      = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    data_collator = data_collator,
)

print(f"Training configuration:")
print(f"  Dataset size     : {len(train_dataset):,} examples")
print(f"  Epochs           : {NUM_EPOCHS}")
print(f"  Batch size       : {BATCH_SIZE} × {GRAD_ACCUMULATION} = {BATCH_SIZE*GRAD_ACCUMULATION} effective")
print(f"  Learning rate    : {LEARNING_RATE}")
print(f"  Scheduler        : cosine with {WARMUP_STEPS} warmup steps")
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUMULATION)
print(f"  Steps per epoch  : ~{steps_per_epoch:,}")
print(f"  Total steps      : ~{steps_per_epoch * NUM_EPOCHS:,}")
print()
print("Starting training... (grab a coffee ☕)")
print("-" * 50)

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time

print("-" * 50)
print(f"✅ Training complete in {elapsed/60:.1f} minutes")
print(f"   Final loss: {trainer.state.log_history[-1].get('loss', 'N/A')}")

Training configuration:
  Dataset size     : 17,088 examples
  Epochs           : 2
  Batch size       : 8 × 2 = 16 effective
  Learning rate    : 0.0002
  Scheduler        : cosine with 100 warmup steps
  Steps per epoch  : ~1,068
  Total steps      : ~2,136

Starting training... (grab a coffee ☕)
--------------------------------------------------


C:\Users\vivik\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## Step 7 · Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt

# Extract loss history
loss_history = [
    (entry['step'], entry['loss'])
    for entry in trainer.state.log_history
    if 'loss' in entry
]

if loss_history:
    steps, losses = zip(*loss_history)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(steps, losses, color='steelblue', linewidth=2)
    ax.fill_between(steps, losses, alpha=0.15, color='steelblue')

    # Mark epoch boundaries
    steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUMULATION)
    for ep in range(1, NUM_EPOCHS):
        ax.axvline(ep * steps_per_epoch, color='tomato',
                   linestyle='--', alpha=0.6, label=f'Epoch {ep} end')

    ax.set_title('Training Loss — SLM Storyteller', fontsize=13, fontweight='bold')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Cross-Entropy Loss')
    ax.grid(True, alpha=0.3)
    if NUM_EPOCHS > 1:
        ax.legend()

    # Annotate final loss
    final_loss = losses[-1]
    ax.annotate(f'Final: {final_loss:.3f}',
                xy=(steps[-1], final_loss),
                xytext=(steps[-1] * 0.85, final_loss + 0.2),
                arrowprops=dict(arrowstyle='->', color='grey'),
                fontsize=10, color='steelblue', fontweight='bold')

    plt.tight_layout()
    plt.savefig('training_loss.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f"Loss interpretation:")
    print(f"  Start : {losses[0]:.3f}  (model is mostly guessing)")
    print(f"  Final : {final_loss:.3f}  ", end="")
    if final_loss < 1.3:
        print("(excellent — strong pattern learning)")
    elif final_loss < 1.8:
        print("(good — solid story generation expected)")
    elif final_loss < 2.5:
        print("(moderate — consider more epochs or larger corpus)")
    else:
        print("(high — model may need more training data)")

## Step 8 · Merge LoRA Weights & Test Generation

Merging folds the LoRA adaptations back into the base model weights. The result is a standard model with no PEFT dependency.

In [ ]:
print("Merging LoRA weights into base model...")
model = model.merge_and_unload()
model.eval()
print("✅ LoRA merged,  model is now standalone")
print(f"   Final parameter count: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def generate_story(
    prompt,
    max_new_tokens = 200,
    temperature    = 0.85,
    top_p          = 0.92,
    top_k          = 50,
    repetition_penalty = 1.15,
):
    """
    Generate a story continuation from a prompt.

    Parameters
    ----------
    prompt             : opening line(s) of the story
    max_new_tokens     : maximum words to generate (approx)
    temperature        : 0.7 = conservative, 1.2 = creative, 1.5+ = wild
    top_p              : nucleus sampling — keeps words summing to this probability
    top_k              : only consider the top-k most likely words at each step
    repetition_penalty : >1.0 discourages repeating the same phrases
    """
    # Wrap prompt in story tag to signal the model
    full_prompt = f"<story>\n{prompt}"

    inputs = tokenizer(
        full_prompt,
        return_tensors = "pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            top_p              = top_p,
            top_k              = top_k,
            repetition_penalty = repetition_penalty,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )

    # Decode and clean
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Remove the prompt prefix and story tags
    story = full_text
    for tag in ['<story>', '</story>']:
        story = story.replace(tag, '')

    # Return only the generated part after the prompt
    if prompt in story:
        story = story[story.index(prompt):]

    return story.strip()


# Test generation
TEST_PROMPTS = [
    "Once upon a time there was a little girl who lived near a dark forest.",
    "The brave knight rode through the enchanted kingdom and discovered",
    "Deep in the ocean where the water was as blue as the sky,",
    "The old wizard looked at the young prince and said:",
    "Every morning the princess climbed to the top of the tower and",
]

print("=" * 65)
print("GENERATION TEST — Trained Model")
print("=" * 65)

for i, prompt in enumerate(TEST_PROMPTS, 1):
    story = generate_story(prompt, max_new_tokens=150, temperature=0.85)
    print(f"\n[{i}] Prompt: \"{prompt}\"")
    print(f"    Output: {story[:300]}")
    if len(story) > 300:
        print("    [...]")
    print()

In [ ]:
# Temperature comparison — same prompt, three different temperatures
# This is exactly what you will demo in Week 3

DEMO_PROMPT = "Once upon a time a brave little prince set off on a journey"

print(f'Prompt: "{DEMO_PROMPT}"')
print()

for temp, label in [(0.7, "Conservative (0.7)"),
                    (1.0, "Balanced   (1.0)"),
                    (1.3, "Creative   (1.3)")]:
    story = generate_story(DEMO_PROMPT, max_new_tokens=100, temperature=temp)
    print(f"[{label}]")
    print(f"  {story[:200]}")
    print()

## Step 9 · Save Locally

Always save locally first. If the Hub push fails, you still have the model.

In [ ]:
LOCAL_SAVE_PATH = "./storyteller_merged"

print(f"Saving merged model to {LOCAL_SAVE_PATH}...")
model.save_pretrained(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)

# Save training metadata
metadata = {
    "base_model"        : BASE_MODEL,
    "tinystories_count" : TINYSTORIES_COUNT,
    "epochs"            : NUM_EPOCHS,
    "lora_r"            : LORA_R,
    "lora_alpha"        : LORA_ALPHA,
    "final_loss"        : trainer.state.log_history[-1].get('loss', 'N/A'),
    "total_examples"    : len(train_dataset),
}

with open(f"{LOCAL_SAVE_PATH}/training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Check saved files
saved_files = list(Path(LOCAL_SAVE_PATH).iterdir())
print(f"\nSaved files:")
for f in sorted(saved_files):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<35} {size_mb:>8.1f} MB")

total_mb = sum(f.stat().st_size for f in saved_files) / 1e6
print(f"\n  Total size: {total_mb:.0f} MB")
print("✅ Model saved locally")

## Step 10 · Push to Hugging Face Hub

You need a Hugging Face account and write access token.

Get your token: **huggingface.co → Settings → Access Tokens → New token (write)**

In [ ]:
'''
from huggingface_hub import login, HfApi

import os
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

HF_REPO_ID = "viviktchaudhary/tiny-slm-storyteller-v1"


api = HfApi()
api.upload_folder(
    folder_path   = "./storyteller_merged",
    repo_id       = HF_REPO_ID,
    repo_type     = "model",
    commit_message= "Upload fine-tuned storyteller SLM",
)
print(f"✅ Done: https://huggingface.co/{HF_REPO_ID}")

'''

## Step 11 · Write the Model Card

A good model card makes your repo look professional and helps participants understand what they are loading in Week 3.

In [ ]:
# ── Build model card ─────────────────────────────────────────────────────────
# Written in parts to avoid nested triple-backtick conflict inside f-strings

final_loss = trainer.state.log_history[-1].get('loss', 'N/A')

metadata = f"""---
language: en
tags:
- text-generation
- storytelling
- fine-tuned
- genai-training
license: apache-2.0
base_model: {BASE_MODEL}
---"""

header = """
# 📖 Tiny SLM Storyteller

A small language model fine-tuned for fairy tale and short story generation.
Built as part of a GenAI & Agentic AI training programme — Week 3: Attention in Action.
"""

training_table = f"""
## Training Details

| Detail | Value |
|--------|-------|
| Base model | `{BASE_MODEL}` |
| Training data | {TINYSTORIES_COUNT:,} TinyStories + Gutenberg fairy tales |
| Training method | LoRA (r={LORA_R}, alpha={LORA_ALPHA}) |
| Epochs | {NUM_EPOCHS} |
| Final loss | {final_loss} |
"""

# Code block is a plain string — no f-string here avoids the nesting conflict
quickstart = """
## Quick Start

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    "REPO_ID_PLACEHOLDER",
    torch_dtype=torch.float32,
)
tokenizer = AutoTokenizer.from_pretrained("REPO_ID_PLACEHOLDER")

def generate_story(prompt, temperature=0.85, max_new_tokens=200):
    inputs = tokenizer(f"<story>\\n{prompt}", return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.92,
        repetition_penalty=1.15,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_story("Once upon a time there was a brave little prince"))
```
"""
# Inject the actual repo ID after the string is built
quickstart = quickstart.replace("REPO_ID_PLACEHOLDER", HF_REPO_ID)

prompts_and_notes = """
## Prompts That Work Well

- `"Once upon a time there was a little girl who lived near a dark forest."`
- `"The brave knight rode through the enchanted kingdom and discovered"`
- `"Deep in the ocean where the water was as blue as the sky,"`
- `"The old wizard looked at the young prince and said:"`

## Notes

- Use `temperature=0.7` for coherent, predictable stories
- Use `temperature=1.2` for more creative and surprising outputs
- Always use `repetition_penalty >= 1.1` to avoid loops
- Model was trained with `<story>` tags — include them in prompts for best results
"""

model_card = metadata

## Step 12 · Verify the Push — Load Fresh from Hub

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"Loading fresh from Hub: {HF_REPO_ID}")
print("This confirms participants will be able to load it in Week 3...")

# Exactly how participants will load it in the Week 3 notebook
verify_model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID,
    torch_dtype = torch.float32,   # float32 for CPU compatibility
)
verify_tok = AutoTokenizer.from_pretrained(HF_REPO_ID)

# Quick generation test
prompt  = "Once upon a time there was a little girl"
inputs  = verify_tok(f"<story>\n{prompt}", return_tensors="pt")

with torch.no_grad():
    output = verify_model.generate(
        **inputs,
        max_new_tokens     = 80,
        temperature        = 0.85,
        top_p              = 0.92,
        repetition_penalty = 1.15,
        do_sample          = True,
        pad_token_id       = verify_tok.eos_token_id,
    )

result = verify_tok.decode(output[0], skip_special_tokens=True)
print(f"\n✅ Hub load verified")
print(f"\nTest generation:")
print(f"  {result[:250]}")
print()
print(f"Share this with participants before Week 3:")
print(f"  model = AutoModelForCausalLM.from_pretrained('{HF_REPO_ID}', torch_dtype=torch.float32)")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{HF_REPO_ID}')")

## ✅ Done — Pre-Session Checklist

## Loading in Week 3 Notebook

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

HF_REPO = "YOUR-USERNAME/tiny-slm-storyteller-v1"   # ← update this

try:
    # Primary: load from Hugging Face Hub
    model     = AutoModelForCausalLM.from_pretrained(HF_REPO, torch_dtype=torch.float32)
    tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
    print(f"✅ Loaded from Hub: {HF_REPO}")
except Exception as e:
    # Fallback: load from local backup (if Hub is unreachable)
    print(f"Hub unavailable ({e}), loading local backup...")
    model     = AutoModelForCausalLM.from_pretrained("./storyteller_merged", torch_dtype=torch.float32)
    tokenizer = AutoTokenizer.from_pretrained("./storyteller_merged")
    print("✅ Loaded from local backup")
```

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

HF_REPO = "viviktchaudhary/tiny-slm-storyteller-v1"

try:
    # Primary: load from Hugging Face Hub
    model     = AutoModelForCausalLM.from_pretrained(HF_REPO, torch_dtype=torch.float32)
    tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
    print(f"✅ Loaded from Hub: {HF_REPO}")
except Exception as e:
    # Fallback: load from local backup (if Hub is unreachable)
    print(f"Hub unavailable ({e}), loading local backup...")
    model     = AutoModelForCausalLM.from_pretrained("./storyteller_merged", torch_dtype=torch.float32)
    tokenizer = AutoTokenizer.from_pretrained("./storyteller_merged")
    print("✅ Loaded from local backup")